# <span style="color:red"> Lab 9 - Predicting the Next Word in Hotel Reviews  </span>

<font size = "4">

- Suppose you’re reading a hotel review and get to the phrase “We thought the room ...”. What word would you expect to come next?

- To answer this question, we'll apply a (very primitive) predictive language model. LLMs use much more sophisticated approaches. Check out DATASCI 340 or [this free textbook](https://web.stanford.edu/~jurafsky/slp3/) for more info.

- We'll estimate a very simple probability distribution from a **trigram** (3-gram) language model, a specific case of the N-gram language model

$\qquad$ <img src="files_lab9/trigram.png" alt="drawing" width="500"/>


<font size = "2">

(quote taken from Chapter 3 of the textbook linked above)

<font size = "4">

Let's read in a dataset containing hotel reviews ([taken from here](https://www.kaggle.com/datasets/datafiniti/hotel-reviews)). The most important column will be "review" which contains text reviewing someone's stay at a hotel:


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hotels = pd.read_csv("files_lab9/Hotel_Reviews.csv")

display(hotels)
display(hotels[["review", "rating"]].head())



### 1. Split the reviews into words

<font size = "4">

Each review starts out as one long string. The code below uses `.str.findall()` to pull out the individual words, which we’ll call "tokens". This is added as an additional column to the DataFrame.

The **regular expression** pattern `[A-Za-z']+` looks for groups of letters and straight apostrophes. The apostrophe lets us treat words like "don't" as a single token. We’ve already written the pattern for you:

In [ ]:
hotels["tokens"] = hotels["review"].str.findall("[A-Za-z']+")
display(hotels[["review", "tokens"]].head())

print("Each entry of 'review' column is a:", type(hotels.loc[0, "review"]))
print("Each entry of 'tokens' column is a:", type(hotels.loc[0, "tokens"]))


### Exercise 1 — Make all words lower-case.

<font size = "4">

We want `Room` and `room` to count as the same word. Try running the next cell to see what `.lower()` does to a string.

Then finish `lower_list`. It takes in a **list** of words (strings) as input, and returns a new list with each word in lowercase. Use `.apply()` to run your function on the series `hotels["tokens"]`, save the result back to that column. Then display the first few rows.

In [ ]:
word = "Room"
print(word)
print(word.lower())
print()
word2 = "HoLiDaY"
print(word2)
print(word2.lower())

In [ ]:

def lower_list(lst):
    # lst is a list of strings
    # return a new list with the same strings, but lowercase
    pass


# Apply lower_list to the "tokens" column, and then overwrite it with the lowercase version

display(hotels[["review", "tokens"]].head())        # all strings in "tokens" should be lowercase now

## 2. Create trigrams

<font size = "4">

A *trigram* is a group of three words in a row. Take the sentence “this was a very bad hotel.” Start with the first three words, then move over one word at a time. You’ll get:

- `("this", "was", "a")`
- `("was", "a", "very")`
- `("a", "very", "bad")`
- `("very", "bad", "hotel")`

We’ll store each trigram as a **tuple** so pandas can use it as an index label when we count how often it appears. The next cell shows how this works for the first hotel review.

In [ ]:
words = hotels["tokens"].iloc[0]  # get list of words from row 0
print(words)
N = len(words)

trigram_list = []
for i in range(N-2):
    trigram = (words[i], words[i+1], words[i+2]) # define trigram as a tuple
    trigram_list.append(trigram)

trigram_list

### Exercise 2 — Build trigrams for every review

<font size = "4">

The example above builds trigrams for the first review. Turn that code into a function called `get_trigrams` that takes a list of words and returns its trigrams in order. If the list has fewer than three words, the function should return an **empty list**.

Use `.apply()` to run `get_trigrams` on the `tokens` column. Save the result in a new column called `trigrams` and display the first few rows.

In [ ]:
def get_trigrams(words):
    pass
# Use .apply() to create the trigrams column and display the first few rows.

### 3. Count the trigrams

#### 3a. Put each trigram on its own row

<font size = "4">

Each row in `hotels["trigrams"]` holds a list of trigrams from one review: 

In [ ]:
hotels["trigrams"].iloc[0]

<font size = "4">

We want each trigram to have its own row. We can use the `.explode()` method to "unravel" each list and gives each element its own row. Compare the output before and after the call below, and note how the number of rows has increased. The index labels repeat because pandas keeps the original review’s row label for each trigram it came from.

In [ ]:
display(hotels["trigrams"])


exploded_trigrams = hotels["trigrams"].explode()
display(exploded_trigrams)

<font size = "4">

What if a review only says “Awesome, Overall.”? Two words aren’t enough to make a trigram, so the list of trigrams is empty: after executing `.explode()` the empty list becomes `nan`:

In [ ]:
print(hotels["review"].loc[46])
print(hotels["trigrams"].loc[46])
print(exploded_trigrams.loc[46])

<font size = "4">

To remove the "nans", we'll use the `.dropna()` method to remove these missing values. Notice how the number of rows decreases:

In [ ]:
print(len(exploded_trigrams))

all_trigrams = exploded_trigrams.dropna()
print(len(all_trigrams))

### 3b. Find the most common trigrams
<font size = "4">

Now that each trigram has its own row, use `.value_counts()` to see how many times each one appears. Save the result as `tri_counts` and display the five most common trigrams.

In [ ]:
# Save the counts as tri_counts and display the five most common trigrams.

### 3c. Turn counts into proportions
<font size = "4">
The counts tell us how many times each trigram appeared. Since our language model uses probabilities, we want to know how often each trigram appears out of all the trigrams we found. For example, if a trigram shows up 5 times out of 100, its proportion is 0.05.

Run `.value_counts()` again with `normalize=True` to get those proportions. Together, they form a probability distribution over the trigrams in our dataset. Save the result as `tri_probs` and display the first five entries.

In [ ]:
# Save the proportions as tri_probs and display the first five entries.


### 3d. Plot the 15 most common trigrams
<font size = "4">

Take the first 15 entries from `tri_probs` and save them as `top_15`. You can use `.head()` or `.iloc`.

Call the `.plot()` method from `top_15` with `kind="bar"` and `figsize=(12, 4)`. Give the plot a title, label the y-axis "Probability", and don't forget to call `plt.show()`!

Are the x-axis labels hard to read vertically? You can use the following command (before plt.show()) to make them more readable: `plt.xticks(rotation=45)`

In [ ]:
# Select the top 15 trigrams.

# Make the bar plot.

# Add a title and y-axis label, then show the plot.

## 4. Predict the next word

### Exercise 4 — What comes after “the room”?

<font size = "4">

Let’s go back to “We thought the room …”. The starter code marks the trigrams that begin with `("the", "room")`. Then a function `check_for_prefix` is written that returns `True` if the first two words of a trigram coincide with the prefix. 

If we use the Series `all_trigrams`, we can simply use `.apply` to get a Boolean mask where each entry is `True` when the trigram starts with "the room":

In [ ]:
prefix = ("the", "room") # predict next word based on these two

def check_for_prefix(trigram):
    # True or False: first two elements of trigram are "the" and "room"
    return trigram[:2] == prefix

bool_mask = all_trigrams.apply(check_for_prefix)

bool_mask

<font size = "4">

Unfortunately, we can't use `bool_mask` with `tri_probs` because the indices don't match. We also can't use `.apply` with `tri_probs` because the trigrams are the **index** of the Series, while the data are the probabilities (floating point numbers).

Since `tri_probs.index` are the actual trigrams, we use a method that is similar to `.apply`. It's called `.map`:

In [ ]:
index_mask = tri_probs.index.map(check_for_prefix)

index_mask

<font size = "4">

Use `index_mask` to extract all the probabilities in `tri_probs` corresponding to trigrams that start with "the room". Name the resulting variable `candidate_trigrams`.

**Hint:** What if `tri_probs` were a NumPy array? It works the same way here.

In [ ]:
# your code here


<font size = "4">

Execute the `.sum()` method from `candidate_trigrams`. Does it sum to 1 (or a number very close to 1)? If not, we don't have a true probability distribution anymore.

Change the values of `candidate_trigrams` so that its values sum to 1 (or close to it), without changing the ratios between them. For example, if one trigram was twice as likely as another before, it should still be twice as likely afterward. After changing them, run `candidate_trigrams.sum()` again to check your work.

**Hint:** The sum of the sequence (1, 1, 1) is 1 + 1 + 1 = 3. How would you change the sequence so that all 3 numbers are still equal, but they sum to 1 instead of 3? How would you change the sequence (1, 2, 4) so that the second number is still double the first, the third number is still double the second, but they sum to 1 instead of 7?

In [ ]:
# your code here



<font size = "4">

The probabilities should be sorted from highest to lowest at this point. Display the 10 most likely trigrams starting with "the room". Print the most likely next word and its probability. The next word is the third word in the top trigram.

In [ ]:
# your code here


<font size = "4">

Repeat the process with the prefix "front desk". What is the most likely next word?

In [1]:
# your code here
